In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

plt.close('all')

ROOT = Path.cwd().parent
processed = ROOT / "data" / "processed"

# ── Load preprocessed artifacts from notebook 02 ──────────────────────────────
X_train_scaled = joblib.load(processed / "X_train_scaled.pkl")
X_test_scaled  = joblib.load(processed / "X_test_scaled.pkl")
X_train_smote  = joblib.load(processed / "X_train_smote.pkl")
X_test         = joblib.load(processed / "X_test.pkl")
y_train_smote  = joblib.load(processed / "y_train_smote.pkl")
y_test         = joblib.load(processed / "y_test.pkl")
print("Artifacts loaded successfully")

# ── Train models ───────────────────────────────────────────────────────────────
# Logistic Regression uses scaled data (linear models require it).
# Tree-based models use unscaled SMOTE data (scale-invariant).

lr_model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)
lr_model.fit(X_train_scaled, y_train_smote)
print("Logistic Regression trained")

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_smote, y_train_smote)
print("Decision Tree trained")

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_smote, y_train_smote)
print("Random Forest trained")

# scale_pos_weight omitted: SMOTE already balanced the training set 1:1.
# Using both SMOTE and scale_pos_weight double-corrects for imbalance.
xgb_model = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    random_state=42, eval_metric='logloss',
)
xgb_model.fit(X_train_smote, y_train_smote)
print("XGBoost trained")

lgbm_model = LGBMClassifier(random_state=42)
lgbm_model.fit(X_train_smote, y_train_smote)
print("LightGBM trained")

cat_model = CatBoostClassifier(verbose=0, random_state=42)
cat_model.fit(X_train_smote, y_train_smote)
print("CatBoost trained")

# ── Evaluate all models ────────────────────────────────────────────────────────
# ROC-AUC and PR-AUC must use continuous probability scores, not binary predictions.
# Using predict() (0/1 output) for roc_auc_score gives a single point on the ROC
# curve and severely understates model quality — always use predict_proba()[:,1].

def eval_model(name, model, X, X_scaled=None):
    """Compute classification metrics using probability scores."""
    X_input = X_scaled if X_scaled is not None else X
    y_pred = model.predict(X_input)
    y_prob = model.predict_proba(X_input)[:, 1]
    return {
        "Model":     name,
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall":    recall_score(y_test, y_pred),
        "F1":        f1_score(y_test, y_pred),
        "ROC-AUC":   roc_auc_score(y_test, y_prob),
        "PR-AUC":    average_precision_score(y_test, y_prob),
    }

results = [
    eval_model("Logistic Regression", lr_model, X_test, X_test_scaled),
    eval_model("Decision Tree",       dt_model, X_test),
    eval_model("Random Forest",       rf_model, X_test),
    eval_model("XGBoost",             xgb_model, X_test),
    eval_model("LightGBM",            lgbm_model, X_test),
    eval_model("CatBoost",            cat_model, X_test),
]

# PR-AUC is the primary ranking metric for imbalanced fraud data.
results_df = (
    pd.DataFrame(results)
    .sort_values("PR-AUC", ascending=False)
    .reset_index(drop=True)
)
print("\nModel comparison (sorted by PR-AUC):")
print(results_df.to_string(index=False))

# ── Best model: detailed report ────────────────────────────────────────────────
print("\nClassification Report — XGBoost:\n")
xgb_pred = xgb_model.predict(X_test)
print(classification_report(y_test, xgb_pred))

# ── Confusion matrix ───────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, xgb_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("XGBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# ── Feature importance ─────────────────────────────────────────────────────────
importance_df = pd.DataFrame({
    'Feature':    X_train_smote.columns,
    'Importance': xgb_model.feature_importances_,
}).sort_values('Importance', ascending=False)

print("\nTop 10 features:\n", importance_df.head(10).to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(10))
plt.title("Top 10 Feature Importance — XGBoost")
plt.tight_layout()
plt.show()

# ── Save artifacts ─────────────────────────────────────────────────────────────
models_dir = ROOT / "models"
models_dir.mkdir(exist_ok=True)

joblib.dump(xgb_model, models_dir / "fraud_detection_model.pkl")
print("\nXGBoost model saved to models/fraud_detection_model.pkl")

reports_dir = ROOT / "reports"
reports_dir.mkdir(exist_ok=True)
results_df.to_csv(reports_dir / "model_results.csv", index=False)
print("Model results saved to reports/model_results.csv")